# ⚡ Лабораторная работа 2. Функции активации

> Практическая часть **главы 2 — «Функции активации»** проекта **Almaz_AI**.

## 🎯 Цель лабораторной работы

На практике увидеть, как разные функции активации преобразуют одинаковые числа и как выбор активации влияет на обучение одной и той же нейронной сети.

Мы исследуем:

- `ReLU`;
- `Sigmoid`;
- `Tanh`;
- `Leaky ReLU`;
- модель **без скрытой функции активации**.

Затем сравним их на знакомой задаче «Светофор».

Главная идея:

```text
Одинаковые данные
      +
Одинаковая архитектура
      +
Одинаковые условия обучения
      ↓
Меняем только Activation
      ↓
Сравниваем результат
```


# 1. Импорт библиотек

Используем:

- `torch` — вычисления и нейронные сети;
- `torch.nn` — готовые функции активации и слои;
- `pandas` — таблицы;
- `matplotlib` — графики.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

print("PyTorch version:", torch.__version__)

# 2. Создаём диапазон входных значений

Чтобы увидеть форму функций, создадим множество значений:

```text
от -6 до +6
```

Это будет наш `x`.


In [ ]:
x = torch.linspace(-6, 6, 400)

print("Количество точек:", len(x))
print("Первые значения:", x[:5])
print("Последние значения:", x[-5:])

# 3. ReLU

Формула:

\[
ReLU(x) = \max(0, x)
\]

То есть:

```text
x < 0  → 0
x >= 0 → x
```


In [ ]:
relu = nn.ReLU()
y_relu = relu(x)

plt.figure(figsize=(9, 5))
plt.plot(x.numpy(), y_relu.numpy())
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("ReLU")
plt.xlabel("x")
plt.ylabel("ReLU(x)")
plt.grid(True)
plt.show()

# 4. Sigmoid

Sigmoid переводит любое число в диапазон:

```text
0 ... 1
```

\[
\sigma(x)=\frac{1}{1+e^{-x}}
\]


In [ ]:
sigmoid = nn.Sigmoid()
y_sigmoid = sigmoid(x)

plt.figure(figsize=(9, 5))
plt.plot(x.numpy(), y_sigmoid.numpy())
plt.axhline(0.5, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Sigmoid")
plt.xlabel("x")
plt.ylabel("Sigmoid(x)")
plt.grid(True)
plt.show()

# 5. Tanh

`Tanh` переводит значения в диапазон:

```text
-1 ... 1
```

и проходит через точку:

```text
x = 0 → y = 0
```


In [ ]:
tanh = nn.Tanh()
y_tanh = tanh(x)

plt.figure(figsize=(9, 5))
plt.plot(x.numpy(), y_tanh.numpy())
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Tanh")
plt.xlabel("x")
plt.ylabel("Tanh(x)")
plt.grid(True)
plt.show()

# 6. Leaky ReLU

В отличие от обычной ReLU отрицательные значения не обнуляются полностью.

Используем:

```text
negative_slope = 0.01
```


In [ ]:
leaky_relu = nn.LeakyReLU(negative_slope=0.01)
y_leaky_relu = leaky_relu(x)

plt.figure(figsize=(9, 5))
plt.plot(x.numpy(), y_leaky_relu.numpy())
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Leaky ReLU")
plt.xlabel("x")
plt.ylabel("LeakyReLU(x)")
plt.grid(True)
plt.show()

# 7. Сравниваем функции на одинаковых значениях

Возьмём несколько конкретных входов:

```text
-5, -2, -1, 0, 1, 2, 5
```

и посмотрим, что возвращает каждая функция.


In [ ]:
sample_x = torch.tensor(
    [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0]
)

comparison_values = pd.DataFrame(
    {
        "x": sample_x.numpy(),
        "ReLU": relu(sample_x).numpy(),
        "Sigmoid": sigmoid(sample_x).numpy(),
        "Tanh": tanh(sample_x).numpy(),
        "Leaky ReLU": leaky_relu(sample_x).numpy(),
    }
)

comparison_values.round(4)

# 8. Общий график

Теперь наложим все функции на один график.

Это удобно для сравнения формы.


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(x.numpy(), y_relu.numpy(), label="ReLU")
plt.plot(x.numpy(), y_sigmoid.numpy(), label="Sigmoid")
plt.plot(x.numpy(), y_tanh.numpy(), label="Tanh")
plt.plot(x.numpy(), y_leaky_relu.numpy(), label="Leaky ReLU")

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.title("Сравнение функций активации")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True)
plt.show()

# 9. Возвращаемся к нейросети «Светофор»

Используем тот же датасет из первой лабораторной работы.

Вход:

```text
[красный, жёлтый, зелёный]
```

Выход:

```text
0 = СТОЯТЬ
1 = ИДТИ
```


In [ ]:
def create_traffic_light_dataset() -> tuple[torch.Tensor, torch.Tensor]:
    """Создаёт полный учебный датасет состояний светофора."""

    features = torch.tensor(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0],
            [0.0, 1.0, 0.0],
            [0.0, 1.0, 1.0],
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 1.0],
            [1.0, 1.0, 0.0],
            [1.0, 1.0, 1.0],
        ],
        dtype=torch.float32,
    )

    targets = torch.tensor(
        [[0.0], [1.0], [0.0], [1.0], [0.0], [0.0], [0.0], [0.0]],
        dtype=torch.float32,
    )

    return features, targets


features, targets = create_traffic_light_dataset()

print(features.shape)
print(targets.shape)

# 10. Универсальная модель с заменяемой активацией

Чтобы сравнение было честным, создадим один класс модели.

Меняться будет только:

```python
activation
```

Архитектура:

```text
3 входа
  ↓
Linear(3 → 4)
  ↓
Activation
  ↓
Linear(4 → 1)
```


In [ ]:
class ActivationNetwork(nn.Module):
    """Нейросеть со сменной функцией активации."""

    def __init__(self, activation: nn.Module | None) -> None:
        """Создаёт сеть с указанной функцией активации."""

        super().__init__()

        self.first_layer = nn.Linear(3, 4)
        self.activation = activation
        self.output_layer = nn.Linear(4, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """Выполняет прямой проход."""

        x = self.first_layer(features)

        if self.activation is not None:
            x = self.activation(x)

        return self.output_layer(x)

# 11. Функции обучения и оценки

Используем одинаковые:

- `BCEWithLogitsLoss`;
- `Adam`;
- `Learning Rate`;
- количество эпох.

Это важно для корректного сравнения.


In [ ]:
def calculate_accuracy(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
) -> float:
    """Вычисляет точность бинарной классификации."""

    model.eval()

    with torch.no_grad():
        logits = model(features)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    return float((predictions == targets).float().mean().item())


def train_model(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
    epochs: int = 500,
    learning_rate: float = 0.05,
) -> tuple[list[float], list[float]]:
    """Обучает модель и возвращает историю Loss и Accuracy."""

    loss_function = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    loss_history: list[float] = []
    accuracy_history: list[float] = []

    for _ in range(epochs):
        model.train()

        logits = model(features)
        loss = loss_function(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_history.append(float(loss.item()))
        accuracy_history.append(
            calculate_accuracy(model, features, targets)
        )

    return loss_history, accuracy_history

# 12. Создаём набор экспериментов

Будем сравнивать:

```text
ReLU
Sigmoid
Tanh
Leaky ReLU
Без активации
```

Для каждой модели перед созданием устанавливаем **одинаковый random seed**.

Так начальные веса будут максимально сопоставимыми.


In [ ]:
activation_factories = {
    "ReLU": lambda: nn.ReLU(),
    "Sigmoid": lambda: nn.Sigmoid(),
    "Tanh": lambda: nn.Tanh(),
    "Leaky ReLU": lambda: nn.LeakyReLU(negative_slope=0.01),
    "Без активации": lambda: None,
}

experiment_results = {}
trained_models = {}

for activation_name, activation_factory in activation_factories.items():
    torch.manual_seed(RANDOM_SEED)

    model = ActivationNetwork(
        activation=activation_factory(),
    )

    loss_history, accuracy_history = train_model(
        model=model,
        features=features,
        targets=targets,
        epochs=500,
        learning_rate=0.05,
    )

    experiment_results[activation_name] = {
        "loss": loss_history,
        "accuracy": accuracy_history,
    }

    trained_models[activation_name] = model

print("Эксперименты завершены.")

# 13. Итоговая таблица

Сравним конечные значения.


In [ ]:
summary_rows = []

for activation_name, history in experiment_results.items():
    summary_rows.append(
        {
            "Activation": activation_name,
            "Final Loss": history["loss"][-1],
            "Final Accuracy": history["accuracy"][-1],
        }
    )

summary_table = pd.DataFrame(summary_rows)

summary_table["Final Loss"] = summary_table["Final Loss"].round(6)
summary_table["Final Accuracy"] = (
    summary_table["Final Accuracy"] * 100
).round(2)

summary_table

# 14. Сравниваем Loss

На одном графике отображаем историю функции потерь всех моделей.


In [ ]:
plt.figure(figsize=(11, 6))

for activation_name, history in experiment_results.items():
    plt.plot(
        history["loss"],
        label=activation_name,
    )

plt.title("Сравнение Loss для разных функций активации")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# 15. Сравниваем Accuracy

Теперь смотрим скорость выхода моделей на правильные решения.


In [ ]:
plt.figure(figsize=(11, 6))

for activation_name, history in experiment_results.items():
    plt.plot(
        history["accuracy"],
        label=activation_name,
    )

plt.title("Сравнение Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True)
plt.show()

# 16. Проверяем итоговые ответы моделей

Посмотрим, как каждая обученная модель классифицирует все 8 состояний.


In [ ]:
def predictions_for_model(
    model: nn.Module,
    features: torch.Tensor,
) -> list[str]:
    """Возвращает текстовые решения модели для всех состояний."""

    model.eval()

    with torch.no_grad():
        probabilities = torch.sigmoid(model(features))
        predictions = (probabilities >= 0.5).int().squeeze()

    return [
        "ИДТИ" if value == 1 else "СТОЯТЬ"
        for value in predictions.tolist()
    ]


prediction_table = pd.DataFrame(
    {
        "Состояние": [
            f"[{int(r)}, {int(y)}, {int(g)}]"
            for r, y, g in features.tolist()
        ],
        "Правильно": [
            "ИДТИ" if value == 1 else "СТОЯТЬ"
            for value in targets.squeeze().int().tolist()
        ],
    }
)

for activation_name, model in trained_models.items():
    prediction_table[activation_name] = predictions_for_model(
        model,
        features,
    )

prediction_table

# 17. Почему модель без активации тоже может работать

Это особенно важный эксперимент.

Можно ожидать:

```text
Без ReLU → сеть не работает
```

Но для нашего светофора это утверждение неверно.

Правило задачи:

```text
ИДТИ = зелёный включён
       И
       красный выключен
```

Эти состояния можно разделить линейной границей.

Поэтому даже линейная модель способна решить задачу.

Это **не означает**, что функции активации не нужны.

Правильный вывод:

> Для простой линейно разделимой задачи нелинейность может быть необязательна.  
> Для сложных нелинейных зависимостей функции активации принципиально расширяют возможности многослойной сети.


# 18. 🧪 Эксперимент — Learning Rate

Попробуй заменить:

```python
learning_rate=0.05
```

на:

```python
0.001
0.01
0.1
0.5
```

и снова сравнить функции.

### Наблюдай

- скорость уменьшения Loss;
- стабильность обучения;
- конечную Accuracy.

Функция активации работает не изолированно — на обучение влияют и другие параметры.


# 19. 🧪 Эксперимент — количество нейронов

Измени:

```python
nn.Linear(3, 4)
```

например на:

```python
nn.Linear(3, 2)
nn.Linear(3, 8)
nn.Linear(3, 16)
```

Не забудь соответственно изменить размер следующего слоя.

Посмотри, меняется ли результат для нашей простой задачи.


# 20. 🧪 Эксперимент — отрицательный наклон Leaky ReLU

Сравни:

```python
nn.LeakyReLU(0.001)
nn.LeakyReLU(0.01)
nn.LeakyReLU(0.1)
```

Посмотри:

- как меняется форма функции;
- меняется ли обучение модели.


# 21. Важное замечание о выходном слое

В нашей сети последний слой возвращает **logits**:

```text
Linear → logits
```

Мы используем:

```python
nn.BCEWithLogitsLoss()
```

Поэтому перед Loss **не добавляем Sigmoid**.

Правильно:

```text
Linear
   ↓
logits
   ↓
BCEWithLogitsLoss
```

Для просмотра вероятности после обучения:

```python
torch.sigmoid(logits)
```

Это важный шаблон, который будет встречаться ещё много раз.


# 22. ❓ Самопроверка

Ответь своими словами:

1. Что делает функция активации?
2. Что ReLU делает с отрицательными числами?
3. В какой диапазон переводит значения Sigmoid?
4. В какой диапазон переводит значения Tanh?
5. Чем Leaky ReLU отличается от ReLU?
6. Что такое нелинейность?
7. Почему несколько Linear-слоёв без активации остаются линейным преобразованием?
8. Что такое Dead ReLU?
9. Что называется насыщением функции?
10. Почему Sigmoid может быть связана с затухающим градиентом?
11. Почему для сравнения моделей мы фиксировали random seed?
12. Почему все модели обучались с одинаковыми Epoch и Learning Rate?
13. Почему модель без активации может решить задачу светофора?
14. Означает ли это, что активации не нужны?
15. Почему перед `BCEWithLogitsLoss` мы не применяем Sigmoid вручную?


# 23. 📌 Что нужно запомнить

Главная идея:

```text
Linear
   ↓
Activation
   ↓
Linear
```

Функция активации позволяет сети строить нелинейные преобразования.

Краткая карта:

```text
ReLU
→ отрицательные значения обнуляются

Sigmoid
→ переводит значения в диапазон 0...1

Tanh
→ переводит значения в диапазон -1...1

Leaky ReLU
→ сохраняет небольшой отрицательный сигнал
```

Но выбор функции зависит от:

- задачи;
- архитектуры;
- места в сети;
- функции потерь;
- других параметров обучения.

---

## ➡️ Следующая глава

**Глава 3. Loss и градиентный спуск**

Теперь мы уже умеем:

```text
получить вход
→ провести его через слои
→ применить Activation
→ получить Prediction
```

Следующий вопрос:

> Как модель измеряет ошибку и понимает, как изменить свои веса?

Для этого понадобятся:

```text
Loss
Gradient
Backpropagation
Gradient Descent
Optimizer
```
